# Run full ERA5 training and both tests in JupyterLab
Use a GPU session. The default inspect mode starts no model. Set ACTION='new_run', then Run All to train in a separate timestamped folder. Prepared data, IDs, date splits, seed and hyperparameters are reused unchanged. The model implementation stays in the Python scripts called below.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, subprocess, os, signal
from IPython.display import display, HTML
BASE=Path.home()/'Desktop/LSTM_RR_Modelling/Model A Lumped (ERA5, 1950-2023)'
PY=BASE.parent/'Model_A/.venv/bin/python'
ACTION='inspect'  # change to 'new_run' to train in a separate folder
assert ACTION in ['inspect','new_run']
if ACTION=='inspect':
    WORK=BASE
elif 'WORK' not in globals() or WORK==BASE:
    assert (BASE/'full_record_coverage_summary.json').exists(), 'Wait for complete preparation.'
    tag=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    WORK=BASE.parent/('Model A Lumped (ERA5, 1950-2023) - rerun '+tag)
    WORK.mkdir()
    for p in BASE.iterdir():
        if p.suffix=='.py' or p.name in ['experiment_config.json','training_basins.txt','unseen_test_basins.txt','split_manifest.csv','stratum_counts.csv','shared_polygon_overlap_screen.csv','coverage_audit.csv','full_record_coverage_summary.json','prior_experiment_hashes.json']:
            shutil.copy2(p,WORK/p.name)
    (WORK/'prepared').symlink_to(BASE/'prepared',target_is_directory=True)
    (WORK/'status.json').write_text(json.dumps({'phase':'prepared'}))
print('Mode:',ACTION)
print('Working folder:',WORK)
print('Original results and prepared data are preserved.')

Mode: inspect
Working folder: /home/pollen26/Desktop/LSTM_RR_Modelling/Model A Lumped (ERA5, 1950-2023)
Original results and prepared data are preserved.


## GPU check

In [2]:
result=subprocess.run([str(PY),'-c',"import torch; assert torch.cuda.is_available(), 'GPU session required'; print(torch.cuda.get_device_name(0))"],capture_output=True,text=True)
print(result.stdout or result.stderr)
if ACTION=='new_run': assert result.returncode==0, 'Use a GPU session.'

NVIDIA H100 80GB HBM3 MIG 1g.10gb



## Train, validate, select checkpoint and run both tests

In [3]:
if ACTION=='new_run':
    if (WORK/'completion.json').exists():
        print('This rerun is complete. Inspect its saved results.')
    else:
        command=[str(PY),'-u',str(WORK/'run_full_record.py')]
        log=WORK/'notebook_pipeline.log'
        with log.open('a') as f:
            process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
            try:
                for line in process.stdout:
                    print(line,end=''); f.write(line); f.flush()
                result=process.wait()
                assert result==0, f'Pipeline failed: inspect {log}'
            except KeyboardInterrupt:
                os.killpg(process.pid,signal.SIGTERM)
                process.wait()
                raise
else:
    print('Inspect mode. Set ACTION to new_run in the setup cell to create a separate rerun.')

Inspect mode. Set ACTION to new_run in the setup cell to create a separate rerun.


## Read saved results

In [4]:
for name in ['full_record_coverage_summary.json','pipeline_status.json','run_seed42/summary.json','run_seed42/temporal_test_2015_latest/summary.json','completion.json']:
    p=WORK/name
    if p.exists():
        print(name)
        print(json.dumps(json.loads(p.read_text()),indent=2))

full_record_coverage_summary.json
{
  "training_donors": 1326,
  "unseen_ids": 100,
  "donors_with_validation": 1320,
  "donors_with_test": 1310,
  "unseen_with_test": 99,
  "first_complete_forcing": "1950-01-01",
  "last_complete_forcing": "2023-01-02",
  "first_observation": "1950-01-01",
  "last_observation": "2023-01-02"
}
pipeline_status.json
{
  "phase": "evaluate_full_record.py"
}
run_seed42/summary.json
{
  "best_epoch": 20,
  "unseen_basins": 100,
  "unseen_scored_basins": 99,
  "median_NSE": 0.5321503927756013,
  "median_KGE": 0.514862213219804
}


To resume after restarting the kernel, set WORK to the existing rerun folder, set ACTION to new_run, and execute only the GPU, run and results cells. Do not create another folder. The pipeline uses last.pt to resume unfinished training and leaves completed runs unchanged.